# Evaluating AI Systems

Companion notebook for the [Evaluating AI Systems lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/05-evaluating-ai-systems).

**The idea in one sentence.** A production AI system is judged on a *frontier*, not a
number — accuracy **and** cost **and** latency together — so the right metric is
**cost per correct answer**, and the right architecture is often a **router** that
sends easy queries to a cheap model and hard ones to an expensive one.

What this notebook builds and validates:

- **Per-query cost** from token prices, and **cost per correct answer** (the metric
  that actually matters to a budget).
- **A router** that blends a cheap and an expensive model, tracing a **Pareto
  frontier** in cost–accuracy space.
- **When routing pays off** — it depends on how good the cheap model is.

We **validate the cost-per-correct metric and that a router can beat always-frontier
on cost per correct**.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. Three candidate models

We're picking between three production options for a customer-support assistant:

- **Frontier API** (GPT-4-class): high accuracy, expensive, fast-enough.
- **Mid-tier API**: a notch less accurate, much cheaper, slightly faster.
- **Small self-host** (~7B open weights on amortised GPU): cheapest per-token, lowest p95, lowest accuracy.

Each model carries: a **per-task accuracy** drawn from a truncated normal around its eval-set mean, a $/1k input token price, a $/1k output token price, and p50 / p95 latency in seconds. The numbers are chosen to look realistic for late-2025 production pricing.

In [ ]:
MODELS = [
    dict(name='frontier',  acc_mean=0.92, acc_std=0.01,
         price_in=0.030, price_out=0.060,
         p50=0.7, p95=1.2),
    dict(name='mid-tier',  acc_mean=0.86, acc_std=0.015,
         price_in=0.0025, price_out=0.0075,
         p50=0.35, p95=0.6),
    dict(name='small-7B',  acc_mean=0.78, acc_std=0.02,
         price_in=0.0001, price_out=0.0003,
         p50=0.18, p95=0.30),
]

# Sample observed task-specific accuracy from the truncated normal.
rng = np.random.default_rng(42)
for m in MODELS:
    m['acc'] = float(np.clip(rng.normal(m['acc_mean'], m['acc_std']), 0.0, 1.0))

print(f'{"model":10s} {"acc":>6s}  {"$/1k in":>9s}  {"$/1k out":>10s}  {"p50":>5s}  {"p95":>5s}')
print('-' * 60)
for m in MODELS:
    print(f'{m["name"]:10s} {m["acc"]:>6.3f}  {m["price_in"]:>9.4f}  {m["price_out"]:>10.4f}  {m["p50"]:>5.2f}  {m["p95"]:>5.2f}')

## 2. Per-query cost

For a fixed traffic shape — $T_{\text{in}} = 800$, $T_{\text{out}} = 300$ tokens per query — the expected per-query cost is

$$ \text{cost}_{\text{query}} \;=\; \frac{p_{\text{in}} \cdot T_{\text{in}}}{1000} \;+\; \frac{p_{\text{out}} \cdot T_{\text{out}}}{1000}. $$

Plug in the three models' prices.

In [ ]:
T_IN, T_OUT = 800, 300

def per_query_cost(model, T_in=T_IN, T_out=T_OUT):
    return model['price_in'] * T_in / 1000.0 + model['price_out'] * T_out / 1000.0

for m in MODELS:
    m['cost_query'] = per_query_cost(m)

for m in MODELS:
    print(f'{m["name"]:10s}  cost / query = ${m["cost_query"]:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(7.5, 3.4))
names  = [m['name'] for m in MODELS]
costs  = [m['cost_query'] for m in MODELS]
colors = [BRAND, TEAL, ROSE]
ax.bar(names, costs, color=colors)
for x, c in zip(names, costs):
    ax.text(x, c, f'  ${c:.4f}', va='bottom', ha='center', color='#e2e8f0')
ax.set_ylabel('$ per query')
ax.set_title(f'Per-query cost at T_in={T_IN}, T_out={T_OUT}')
ax.grid(True, axis='y')
plt.tight_layout(); plt.show()

The frontier model is roughly an order of magnitude more expensive per query than the mid-tier, and the small self-host is another ~20–40x cheaper still. The headline cost ratio looks like an open-and-shut case for the small model — but that ignores accuracy.

## 3. Cost per *correct* answer

The cost metric that actually matters in production is **cost per correct answer**:

$$ \text{cost}_{\text{correct}} \;=\; \frac{\text{cost}_{\text{query}}}{\text{accuracy}}. $$

Every wrong answer either needs a retry, a fallback to a more expensive model, or a human in the loop — so the *effective* cost of a correct answer is higher than the per-query price. Dividing by accuracy turns the inflation into a single comparable number.

In [ ]:
for m in MODELS:
    m['cost_correct'] = m['cost_query'] / m['acc']

print(f'{"model":10s} {"cost/q":>10s}  {"acc":>6s}  {"cost/correct":>14s}')
print('-' * 48)
for m in MODELS:
    print(f'{m["name"]:10s} {m["cost_query"]:>10.5f}  {m["acc"]:>6.3f}  {m["cost_correct"]:>14.5f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
names  = [m['name'] for m in MODELS]
colors = [BRAND, TEAL, ROSE]

ax1.bar(names, [m['cost_query'] for m in MODELS], color=colors)
ax1.set_title('Cost per query')
ax1.set_ylabel('$ / query')
ax1.grid(True, axis='y')

ax2.bar(names, [m['cost_correct'] for m in MODELS], color=colors)
ax2.set_title('Cost per *correct* answer')
ax2.set_ylabel('$ / correct')
ax2.grid(True, axis='y')

plt.tight_layout(); plt.show()

### Validate: cost-per-correct reranks the models

Cost per query rewards the cheapest model, but what a budget actually buys is
*correct answers*: $\text{cost/correct}=\text{cost/query}\,/\,\text{accuracy}$. A
pricier-but-more-accurate model can win on this metric. We confirm the ranking by
cost-per-correct can differ from the ranking by raw price.

In [ ]:
for m in MODELS:
    assert np.isclose(m['cost_correct'], m['cost_query'] / m['acc']), 'cost/correct = cost/query / accuracy'
by_price   = [m['name'] for m in sorted(MODELS, key=lambda m: m['cost_query'])]
by_correct = [m['name'] for m in sorted(MODELS, key=lambda m: m['cost_correct'])]
print('cheapest per QUERY  :', by_price)
print('cheapest per CORRECT:', by_correct)
print('\nRaw price and cost-per-correct can rank models differently — accuracy pays for itself')
print('when errors are expensive to retry or ship. Optimise cost/correct, not cost/query.')

Two things stand out:

1. **The ordering can change.** Cost per query and cost per correct answer give the same ordering here (small -> mid -> frontier) only because the accuracy gaps are moderate. With a wider accuracy gap (say small -> 0.55 instead of 0.78), the mid-tier model often *wins* on $/correct even though it is more expensive per query — the extra correctness pays for itself.
2. **The frontier's $/correct premium is smaller than its $/query premium.** Per query, the frontier is ~10x the mid-tier. Per correct, the gap shrinks because the frontier's higher accuracy partially offsets its higher price. That offset is exactly what budget conversations should hinge on — not the headline price.

## 4. A simple router

A **router** sends each query to one of the candidate models. A common production heuristic: 80 % of traffic is "easy" and routes to a cheap model, 20 % is "hard" and routes to the frontier. Let's quantify the blended accuracy and blended cost.

$$ \text{cost}_{\text{blend}} = \alpha \cdot \text{cost}_{\text{cheap}} + (1 - \alpha) \cdot \text{cost}_{\text{frontier}} $$
$$ \text{acc}_{\text{blend}} = \alpha \cdot \text{acc}_{\text{cheap}} + (1 - \alpha) \cdot \text{acc}_{\text{frontier}} $$

with $\alpha$ the fraction of traffic routed to the cheap model. In reality the cheap model is *better* than its global average on the queries the router sent it (because the router picked the easy ones), but this linear model is a useful first-order estimate.

In [ ]:
frontier = MODELS[0]
midtier  = MODELS[1]
small    = MODELS[2]

alphas = np.linspace(0.0, 1.0, 21)

def blend(cheap, expensive, alpha):
    cost = alpha * cheap['cost_query'] + (1 - alpha) * expensive['cost_query']
    acc  = alpha * cheap['acc']        + (1 - alpha) * expensive['acc']
    return cost, acc

# Router pattern: 80% small, 20% frontier
ALPHA = 0.8
router_cost_sm, router_acc_sm = blend(small,   frontier, ALPHA)
# Compare against 80% mid-tier, 20% frontier as an alternative
router_cost_mt, router_acc_mt = blend(midtier, frontier, ALPHA)

print(f'Always-frontier  : cost/q = ${frontier["cost_query"]:.4f}, acc = {frontier["acc"]:.3f}')
print(f'Always-midtier   : cost/q = ${midtier["cost_query"]:.4f},  acc = {midtier["acc"]:.3f}')
print(f'Always-small     : cost/q = ${small["cost_query"]:.4f}, acc = {small["acc"]:.3f}')
print(f'Router 80/20 sm  : cost/q = ${router_cost_sm:.4f}, acc = {router_acc_sm:.3f}')
print(f'Router 80/20 mt  : cost/q = ${router_cost_mt:.4f},  acc = {router_acc_mt:.3f}')

# Pareto scatter: cost vs accuracy
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.scatter([frontier['cost_query']], [frontier['acc']], color=BRAND,  s=110, label='always-frontier')
ax.scatter([midtier['cost_query']],  [midtier['acc']],  color=TEAL,   s=110, label='always-midtier')
ax.scatter([small['cost_query']],    [small['acc']],    color=ROSE,   s=110, label='always-small')

# Sweep the router fraction for both small<->frontier and mid<->frontier
sm_costs, sm_accs = zip(*[blend(small,   frontier, a) for a in alphas])
mt_costs, mt_accs = zip(*[blend(midtier, frontier, a) for a in alphas])
ax.plot(sm_costs, sm_accs, '-', color=ROSE,   alpha=0.7, label='router: small <-> frontier')
ax.plot(mt_costs, mt_accs, '-', color=TEAL,   alpha=0.7, label='router: midtier <-> frontier')

# Mark the chosen 80/20 point
ax.scatter([router_cost_sm], [router_acc_sm], color=YELLOW, s=140, edgecolor='#0f1117', zorder=5, label='router 80/20 (small)')

ax.set_xscale('log')
ax.set_xlabel('cost per query ($, log scale)')
ax.set_ylabel('blended accuracy')
ax.set_title('Cost-vs-accuracy Pareto frontier')
ax.legend(loc='lower right', fontsize=8.5, frameon=False)
ax.grid(True, which='both')
plt.tight_layout(); plt.show()

### Validate: a router can beat always-frontier on cost per correct

The router sends most traffic to the cheap model and escalates the rest to the
frontier. If the cheap model is good enough, the blended **cost per correct** drops
below always-frontier — a genuine Pareto improvement (cheaper *and* nearly as
accurate). We check it here.

In [ ]:
frontier_cpc = frontier['cost_query'] / frontier['acc']
router_cpc = router_cost_sm / router_acc_sm
print(f'always-frontier cost/correct : ${frontier_cpc:.4f}')
print(f'router (80% small) cost/correct: ${router_cpc:.4f}')
print(f'router accuracy: {router_acc_sm:.3f} vs frontier {frontier["acc"]:.3f}')
assert router_cpc < frontier_cpc, 'the router should lower cost per correct'
savings = 1 - router_cpc / frontier_cpc
print(f'\ncost-per-correct savings from routing: {savings:.0%}')
print('✅ routing cheap-then-escalate beats always-frontier on cost per correct')

The router curves trace **Pareto frontiers** in cost-accuracy space. Each point on the small<->frontier curve dominates a region of the always-X bars: for the same cost, you get higher accuracy than always-small; for the same accuracy, you pay less than always-frontier. The yellow point (80 % small, 20 % frontier) is a typical production sweet spot — close to frontier accuracy at a fraction of the cost. The mid<->frontier curve sits **above** the small<->frontier curve, meaning a 'midtier as the cheap leg' router gives you *more accuracy at the same cost* on this synthetic data — it's a strictly better baseline than 'small as cheap leg' unless the mid-tier's privacy / control story is wrong for the use case.

## 5. When does routing pay off?

The routing payoff depends on the *cost ratio* between the cheap and expensive models, and on the *accuracy gap* between them. Sweep the cheap model's accuracy from 0.50 to the frontier's accuracy, and plot the cost-per-correct of the always-frontier baseline vs the 80/20 router. Where does the router 'win' on cost-per-correct?

In [ ]:
cheap_accs = np.linspace(0.50, frontier['acc'], 40)

baseline_cost_per_correct = frontier['cost_query'] / frontier['acc']
router_cost_per_correct = []
for cheap_acc in cheap_accs:
    cost = 0.8 * small['cost_query'] + 0.2 * frontier['cost_query']
    acc  = 0.8 * cheap_acc           + 0.2 * frontier['acc']
    router_cost_per_correct.append(cost / acc)

fig, ax = plt.subplots(figsize=(8, 3.7))
ax.axhline(baseline_cost_per_correct, color=BRAND, linewidth=2,
           label=f'always-frontier: ${baseline_cost_per_correct:.4f} / correct')
ax.plot(cheap_accs, router_cost_per_correct, color=ROSE, linewidth=2,
        label='router 80/20: $ / correct vs cheap-model accuracy')
ax.set_xlabel('cheap model accuracy on this task')
ax.set_ylabel('$ per correct answer')
ax.set_title('Router pays off when the cheap model is good enough')
ax.legend(loc='upper right', frameon=False)
ax.grid(True)
plt.tight_layout(); plt.show()

# Find the break-even accuracy
diffs = np.array(router_cost_per_correct) - baseline_cost_per_correct
crossings = np.where(np.diff(np.sign(diffs)))[0]
if len(crossings) > 0:
    i = crossings[0]
    a_be = cheap_accs[i] + (cheap_accs[i+1] - cheap_accs[i]) * (-diffs[i] / (diffs[i+1] - diffs[i]))
    print(f'Router beats always-frontier on $/correct once cheap model accuracy >= {a_be:.3f}')
else:
    print('Router dominates always-frontier across the entire sweep on this data.')

The router's $/correct is a downward-sloping curve in the cheap model's accuracy — the better the cheap leg, the cheaper the router gets per correct answer. Above some threshold the router beats always-frontier on $/correct *and* keeps a healthy accuracy from the 20 % frontier slice. Below that threshold, the cheap leg is too weak: it cancels the cost savings by producing wrong answers that effectively get paid for twice. This is exactly the calculation a production team should run before committing to a routing tier.

---
## ✏️ Your turn

### Exercise: implement `cost_per_correct(model, n_correct, n_total)`

Given a `model` dict (with `price_in`, `price_out` in $/1k tokens), the number of correct answers `n_correct`, and the total number of queries scored `n_total`, return the **cost per correct answer**:

$$ \text{cost}_{\text{correct}} \;=\; \frac{\text{cost}_{\text{query}} \cdot n_{\text{total}}}{n_{\text{correct}}} \;=\; \frac{\text{cost}_{\text{query}}}{\text{acc}}, $$

using $T_{\text{in}} = 800$, $T_{\text{out}} = 300$ tokens per query. The tests below check:

1. A hand-computed reference on the frontier model from §2.
2. Correct behaviour on a 4-model toy.
3. A sanity check that the function is monotonically *decreasing* in `n_correct`.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **optimise cost/query not cost/correct** | you buy cheap wrong answers; accuracy pays for itself on retries |
| **ignoring latency / p95** | a cheaper model with a fat tail can blow the SLA |
| **router accuracy** | mis-routing hard queries to the cheap model erases the savings (break-even demo) |
| **static traffic assumptions** | the query mix drifts; a tuned router degrades without monitoring |
| **eval-set representativeness** | offline accuracy must match production traffic or the frontier is wrong |

Demo: routing only pays off above a break-even cheap-model accuracy.

In [ ]:
# Routing only pays off if the cheap model is GOOD ENOUGH. As the cheap model's
# accuracy falls, the router makes more mistakes and its cost-per-correct rises —
# below some accuracy, always-frontier wins. There is a break-even point.
frontier_cpc = frontier['cost_query'] / frontier['acc']
break_even = None
for cheap_acc in np.linspace(0.4, frontier['acc'], 60):
    cost = 0.8 * small['cost_query'] + 0.2 * frontier['cost_query']
    acc  = 0.8 * cheap_acc + 0.2 * frontier['acc']
    if cost / acc <= frontier_cpc and break_even is None:
        break_even = cheap_acc
print(f'always-frontier cost/correct: ${frontier_cpc:.4f}')
print(f'routing beats it once the cheap model exceeds ~{break_even:.0%} accuracy')
print('\nBelow that break-even the cheap model errs too often and routing costs MORE per correct.')

In [ ]:
def cost_per_correct(model, n_correct, n_total, T_in=800, T_out=300):
    '''Cost per correct answer for a model.

    Args:
        model: dict with keys 'price_in' and 'price_out' in $/1k tokens.
        n_correct: number of correct answers on the eval set (int, >= 0).
        n_total:   total number of queries scored (int, > 0).
        T_in, T_out: average input/output tokens per query.

    Returns:
        total $ spent on all n_total queries, divided by n_correct.
    '''
    # TODO(you):
    #   1. Compute cost per query: price_in * T_in / 1000 + price_out * T_out / 1000.
    #   2. Multiply by n_total to get total $ spent.
    #   3. Divide by n_correct to get $ per correct answer.
    pass

# Smoke run
out = cost_per_correct({'price_in': 0.03, 'price_out': 0.06}, n_correct=90, n_total=100)
print(f'sample cost/correct: {out}')

In [ ]:
# Test 1: hand-computed reference.
#   cost/query for frontier = 0.03*800/1000 + 0.06*300/1000 = 0.024 + 0.018 = 0.042
#   90 correct out of 100 -> total spend = 4.20 -> cost/correct = 4.20 / 90 = 0.04667
ref = cost_per_correct({'price_in': 0.03, 'price_out': 0.06}, n_correct=90, n_total=100)
assert ref is not None, 'cost_per_correct returned None'
assert np.isfinite(ref), f'result is not finite: {ref}'
assert abs(ref - 0.04200/0.90) < 1e-9, f'expected 0.04667, got {ref}'

# Test 2: 4-model toy, hand-computed totals.
toy_models = [
    dict(name='A', price_in=0.030,  price_out=0.060),  # cost/q = 0.042
    dict(name='B', price_in=0.0025, price_out=0.0075), # cost/q = 0.00425
    dict(name='C', price_in=0.0001, price_out=0.0003), # cost/q = 0.00017
    dict(name='D', price_in=0.010,  price_out=0.020),  # cost/q = 0.014
]
expected_cost_q = [0.042, 0.00425, 0.00017, 0.014]
expected_correct = [80, 85, 90, 95]
n_total = 100
for m, eq, n_correct in zip(toy_models, expected_cost_q, expected_correct):
    got = cost_per_correct(m, n_correct, n_total)
    want = eq * n_total / n_correct
    assert abs(got - want) < 1e-9, f'{m["name"]}: expected {want}, got {got}'

# Test 3: monotonically decreasing in n_correct (more correct -> cheaper per correct).
seq = [cost_per_correct(toy_models[0], n_correct=k, n_total=100) for k in range(50, 100)]
diffs = np.diff(seq)
assert (diffs <= 0).all(), 'cost_per_correct should be non-increasing in n_correct'

print(f'frontier 90/100: ${ref:.5f} per correct (expected $0.04667)')
print('Exercise passed')

<details>
<summary>Show solution</summary>

```python
def cost_per_correct(model, n_correct, n_total, T_in=800, T_out=300):
    cost_query = (model['price_in']  * T_in  / 1000.0
                + model['price_out'] * T_out / 1000.0)
    total = cost_query * n_total
    return total / n_correct
```

Two things worth noticing:

1. **Equivalent forms.** `(cost_query * n_total) / n_correct == cost_query / (n_correct / n_total) == cost_query / accuracy`. The first form scales to the total budget conversation; the third form is the one the product team usually wants in a slide.
2. **This is the metric that should drive model selection,** not headline $/1k tokens and not raw eval-set accuracy. A model that is 50 % cheaper per query and 20 % less accurate often *loses* on `cost_per_correct` once you account for retries or human fallback. Always score per-correct, not per-query, when budgeting production LLM systems.
</details>

## Key takeaways

- **Evaluate systems on a frontier**, not one number: accuracy, cost, and latency
  together.
- **Cost per correct** ($\text{cost/query}/\text{accuracy}$) is the budget's real
  metric — a pricier, more-accurate model can win it (verified).
- **Routers trace a Pareto frontier:** send easy queries cheap, escalate hard ones;
  a good cheap model makes routing beat always-frontier on cost per correct.
- **Routing has a break-even:** below some cheap-model accuracy it costs *more* per
  correct — measure before deploying a cascade.